In [60]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

In [61]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

Cargar archivos csv

In [62]:
df = pd.read_csv(root / "csv/ejercicios.csv", encoding="utf-8")

In [63]:
df_sin_metadatos = df[df.aviso == "sin metadatos"]
df = df[df.aviso.isna()]

In [64]:
df_sin_metadatos

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso


In [65]:
df.sample()

,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
4342,Química,Química,2019,Reserva 2,Ejercicio 4. Opción B,Reactividad Orgánica,QUÍMICA. 2019. RESERVA 2. EJERCICIO 4. OPCIÓN B,Complete las siguientes reacciones e indique a...,source_pdf_files/Química/reactividad_orgánic...,enunciados_ejercicios/Química/reactividad_org...,NaN


Comprobar si coinciden los valores de asignatura y asignatura_carpeta

In [66]:
df.asignatura = df.asignatura.str.normalize("NFC")
df.asignatura_carpeta = df.asignatura_carpeta.str.normalize("NFC")

In [67]:
if all(df.asignatura == df.asignatura_carpeta):
    df.drop("asignatura_carpeta", axis = 1, inplace=True)

In [68]:
df.asignatura.unique()

<ArrowStringArray>
['Física', 'Mates CCSS', 'Mates II', 'Química']
Length: 4, dtype: str

In [69]:
df.convocatoria.unique()

<ArrowStringArray>
[             'Junio',          'Reserva 2',          'Reserva 3',
          'Reserva 4',         'Septiembre',          'Reserva 1',
 'Reserva1 Ejercicio',              'Julio',           'Ponencia',
             'Modelo',                  nan]
Length: 11, dtype: str

In [70]:
df[df.convocatoria.isna()]

,asignatura,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen,aviso
2556,Mates II,2017,NaN,NaN,Matrices y determinantes,MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE...,Sea A una matriz 3x3 tal que det(2A)8.\na) ¿C...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...,NaN


In [71]:
print(*df[df.convocatoria.isna()].info_ejercicio, sep = "\n")

MATEMÁTICAS II. 2017. RESERR VE A S 3 .O E JLE RUC CIC II O Ó 3 .N O PCIÓN B.


In [72]:
print(*df[df.convocatoria.isna()].archivo, sep = "\n")

source_pdf_files/Mates II/matrices_determinantes/2017 - Matrices y determinantes.pdf


In [73]:
df.loc[df.convocatoria.isna(), ["convocatoria", "ejercicio"]] = ["Reserva 3", "Ejercicio 3, opción B"]

In [95]:
df[["asignatura", "tema"]].drop_duplicates()

,asignatura,tema
0,Física,campo eléctrico y magnético
121,Física,campo gravitatorio
244,Física,física cuántica y nuclear
362,Física,ondas
449,Física,óptica geométrica
486,Mates CCSS,contraste de hipótesis
545,Mates CCSS,distribución binomial
550,Mates CCSS,funciones ccss
806,Mates CCSS,inferencia estadística
1078,Mates CCSS,matrices y determinantes


In [77]:
df["tema"] = df["tema"].str.normalize("NFC").str.strip().str.lower()

In [78]:
df.loc[df.tema == "conf. electrónica", "tema"] = "configuración electrónica" 

In [79]:
print(df[df.ejercicio.isna()].info_ejercicio)

Series([], Name: info_ejercicio, dtype: str)


In [80]:
df.loc[df.convocatoria == "Reserva1 Ejercicio", ["convocatoria", "ejercicio"]] = ["Reserva 1", "Ejercicio 1"]

Año a integer


In [81]:
df.año = df.año.astype(int)

Pasar "Opción" a minúsculas


In [82]:
df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción")
                                      .replace(".", ",") if pd.notna(x) else x)


Eliminar datos de gravedad de enunciados de física

In [83]:
pattern = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
mask = df.asignatura == "Física"
df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(pattern, "", regex=True)

Normalizar texto

In [84]:
# df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto)
#                                       if pd.notna(texto) else texto)

Eliminar espacios múltiples de los enunciados.

In [85]:
df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto)
                                      .strip() if pd.notna(texto) else texto)

Eliminar "," al final del nonmbre de ejercicio

In [86]:
df.ejercicio = df.ejercicio.str.strip(",")

## Crear columna id

Tras la limpieza completa, creo el ID

In [87]:
def create_id(row):
    partes = [row["asignatura"], row["tema"], row["año"], row["convocatoria"]]
    if pd.notna(row["ejercicio"]):
        partes.append(row["ejercicio"])
    # se usa str() para convertir año (int) en string
    id_string = "_".join(str(p) for p in partes).lower()

    # quitar acentos: descomponer y eliminar los caracteres combinantes
    id_string = unicodedata.normalize("NFKD", id_string)
    id_string = "".join(c for c in id_string if not unicodedata.combining(c))

    # dejar solo letras, números y _ (quita comas, puntos, espacios...)
    id_string = re.sub(r"[^a-z0-9]+", "_", id_string).strip("_")
    return id_string

In [88]:
df["id"] = df.apply(lambda row: create_id(row), axis = 1)

Reorganizar las columnas

In [89]:
df = df[["id"] + list(df.columns[:-2])]

## Exportar el dataframe limpio

In [90]:
df = (df.sort_values(["asignatura", "tema", "año", "convocatoria", "ejercicio"],
               ascending = [True, True, False, True, True])
               .reset_index(drop = True)
)

In [91]:
df.sample(5)

,id,asignatura,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
1349,mates_ccss_probabilidad_2015_reserva_4_ejercic...,Mates CCSS,2015,Reserva 4,"Ejercicio 3, Opcion B",probabilidad,SOCIALES II. 2015 RESERVA 4 EJERCICIO 3. OPCION B,Una enfermedad puede estar provocada por solo ...,source_pdf_files/Mates CCSS/probabilidad/2015 ...,enunciados_ejercicios/Mates CCSS/probabilidad/...
759,mates_ccss_funciones_ccss_2008_septiembre_ejer...,Mates CCSS,2008,Septiembre,"Ejercicio 2, opción A",funciones ccss,SOCIALES II. 2008 SEPTIEMBRE. EJERCICIO 2. OPC...,3\na) Halle la ecuación de la recta tangente a...,source_pdf_files/Mates CCSS/funciones_ccss/200...,enunciados_ejercicios/Mates CCSS/funciones_ccs...
3275,quimica_configuracion_electronica_2002_reserva...,Química,2002,Reserva 2,"Ejercicio 3, opción B",configuración electrónica,QUÍMICA. 2002. RESERVA 2. EJERCICIO 3. OPCIÓN B,"Dados los elementos A (Z = 13), B (Z = 9) y C ...",source_pdf_files/Química/conf._electrónica/2...,enunciados_ejercicios/Química/conf._electrón...
615,mates_ccss_funciones_ccss_2020_reserva_2_ejerc...,Mates CCSS,2020,Reserva 2,Ejercicio B3,funciones ccss,SOCIALES II. 2020. RESERVA 2. EJERCICIO B3,a) (1’2 puntos) Calcule la función derivada de...,source_pdf_files/Mates CCSS/funciones_ccss/202...,enunciados_ejercicios/Mates CCSS/funciones_ccs...
67,fisica_campo_electrico_y_magnetico_2020_reserv...,Física,2020,Reserva 2,Ejercicio 2,campo eléctrico y magnético,FISICA. 2020. RESERVA 2. EJERCICIO 2,"a) Un conductor rectilíneo de longitud L, por ...",source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...


In [92]:
df.to_csv(root/"csv/ejercicios_clean.csv", index = False, encoding="utf-8")

In [93]:
df = pd.read_csv(root / "csv/ejercicios_clean.csv", encoding="utf-8")

In [94]:
df.sample(3)

,id,asignatura,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
2867,mates_ii_sistemas_de_ecuaciones_lineales_2016_...,Mates II,2016,Reserva 3,"Ejercicio 3, opción B",sistemas de ecuaciones lineales,MATEMÁTICAS II. 2016. RESERVA 3. EJERCICIO 3. ...,De los datos recabados en un informe sobre los...,source_pdf_files/Mates II/sistemas_de_ecuacion...,enunciados_ejercicios/Mates II/sistemas_de_ecu...
3766,quimica_formulacion_2015_reserva_1_ejercicio_1...,Química,2015,Reserva 1,"Ejercicio 1, opción B",formulación,QUÍMICA. 2015. RESERVA 1. EJERCICIO 1. OPCIÓN B,Formule o nombre los siguientes compuestos: a)...,source_pdf_files/Química/formulación/2015 - ...,enunciados_ejercicios/Química/formulación/pn...
2820,mates_ii_sistemas_de_ecuaciones_lineales_2023_...,Mates II,2023,Reserva 4,Ejercicio 6,sistemas de ecuaciones lineales,MATEMÁTICAS II. 2023. RESERVA 4. EJERCICIO 6,Una plataforma de streaming se especializa en ...,source_pdf_files/Mates II/sistemas_de_ecuacion...,enunciados_ejercicios/Mates II/sistemas_de_ecu...
